# Interactive SmolVLA

Same workflow as `interactive_lava.ipynb`, but the policy is a LeRobot SmolVLA
checkpoint. Because SmolVLA needs PyTorch + numpy 2 (the `lerobotenv`
env) while LanguageTable needs JAX + numpy 1 (`ltvenv`), the model runs in a
subprocess and the notebook talks to it over TCP. The wrapper class below
hides that — it exposes the same `predict(goals, obs_list, active_mask)`
shape as `LAVAPolicy`, so the rest of the notebook (and the `Recorder`) is
identical to the LAVA one.

Run this notebook with the `langtable` (ltvenv) kernel.

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/projects/language-table"))

import nest_asyncio; nest_asyncio.apply()
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from language_table.environments.language_table import LanguageTable
from language_table.environments.blocks import LanguageTableBlockVariants
from language_table.environments.rewards.block2block import BlockToBlockReward
from language_table.lamer.state_to_text import state_to_text

## SmolVLA policy wrapper

Spawns `training/eval/lerobot_policy_server.py` in `lerobotenv`, waits
for it to bind, and provides `predict` / `reset` / `close`. Mirrors the
`LAVAPolicy` API used in `interactive_lava.ipynb` so the rest of the
notebook is policy-agnostic.

`reset(num_envs=1)` is called between high-level commands to clear SmolVLA's
internal action chunk buffer (otherwise the next command would start with
queued actions from the previous one).

In [ ]:
import atexit, pickle, socket, struct, subprocess, time

REPO = os.path.expanduser("~/projects/language-table")
LEROBOT_PYTHON = f"{REPO}/lerobotenv/bin/python"
SERVER_SCRIPT  = f"{REPO}/training/eval/lerobot_policy_server.py"

_HEADER_FMT = "!I"
_HEADER_SIZE = struct.calcsize(_HEADER_FMT)

def _send(sock, obj):
    payload = pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)
    sock.sendall(struct.pack(_HEADER_FMT, len(payload)) + payload)

def _recv(sock):
    head = b""
    while len(head) < _HEADER_SIZE:
        chunk = sock.recv(_HEADER_SIZE - len(head))
        if not chunk:
            raise ConnectionError("socket closed mid-header")
        head += chunk
    (n,) = struct.unpack(_HEADER_FMT, head)
    buf = b""
    while len(buf) < n:
        chunk = sock.recv(n - len(buf))
        if not chunk:
            raise ConnectionError("socket closed mid-payload")
        buf += chunk
    return pickle.loads(buf)


class SmolVLAPolicy:
    def __init__(self, checkpoint_path,
                 host="127.0.0.1", port=50100,
                 server_log="/tmp/smolvla_interactive.log",
                 ready_timeout=300.0):
        self.host, self.port = host, port
        self.proc, self.sock = None, None
        self._spawn_server(checkpoint_path, server_log, ready_timeout)
        self._connect()
        atexit.register(self.close)

    def _spawn_server(self, checkpoint, log_path, timeout):
        log = open(log_path, "w")
        self.proc = subprocess.Popen(
            [LEROBOT_PYTHON, "-u", SERVER_SCRIPT,
             "--checkpoint_path", checkpoint,
             "--host", self.host, "--port", str(self.port)],
            stdout=log, stderr=subprocess.STDOUT,
        )
        deadline = time.time() + timeout
        while time.time() < deadline:
            with open(log_path) as f:
                if "Policy server listening" in f.read():
                    print(f"SmolVLA server up (pid={self.proc.pid}, log={log_path})")
                    return
            if self.proc.poll() is not None:
                with open(log_path) as f:
                    raise RuntimeError(f"server died:\n{f.read()[-2000:]}")
            time.sleep(1.0)
        self.proc.terminate()
        raise RuntimeError(f"server not ready within {timeout}s; see {log_path}")

    def _connect(self):
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.sock.connect((self.host, self.port))

    def reset(self, num_envs=1):
        _send(self.sock, {"method": "reset"})
        resp = _recv(self.sock)
        if resp.get("status") != "ok":
            raise RuntimeError(f"reset failed: {resp.get('error_message')}")

    def predict(self, goals, obs_list, active_mask):
        actions = []
        for goal, obs, active in zip(goals, obs_list, active_mask):
            if not active:
                actions.append(np.zeros(2, dtype=np.float32))
                continue
            rgb = obs["rgb"]
            if rgb.dtype != np.uint8:
                rgb = (rgb * 255).clip(0, 255).astype(np.uint8)
            state = np.asarray(
                obs.get("effector_translation", np.zeros(2, dtype=np.float32)),
                dtype=np.float32,
            )
            _send(self.sock, {
                "method": "action",
                "rgb": rgb.tolist(),
                "state": state.tolist(),
                "instruction": goal,
            })
            resp = _recv(self.sock)
            if resp.get("status") != "ok":
                raise RuntimeError(f"action failed: {resp.get('error_message')}")
            actions.append(np.asarray(resp["action"], dtype=np.float32))
        return actions

    def close(self):
        if self.sock is not None:
            try:
                _send(self.sock, {"method": "close"})
                _recv(self.sock)
            except Exception:
                pass
            try: self.sock.close()
            except Exception: pass
            self.sock = None
        if self.proc is not None and self.proc.poll() is None:
            self.proc.terminate()
            try: self.proc.wait(timeout=10)
            except subprocess.TimeoutExpired: self.proc.kill()
            self.proc = None

    def __del__(self):
        self.close()

## Build env + load policy

The first call to `SmolVLAPolicy(...)` blocks for ~10–60 s while the server
process loads the SmolVLM2 weights onto GPU. The cell is idempotent — if
you re-run it, the previous server is closed first, so you can edit the
checkpoint path or reseed the env without restarting the kernel.

In [ ]:
# Idempotent: if a policy is already alive in this kernel, tear it down
# first so we can re-run this cell without a port-bind conflict.
if "policy" in globals():
    try:
        policy.close()
        print("Closed prior SmolVLA server.")
    except Exception as e:
        print(f"(warn) prior close failed: {e}")

env = LanguageTable(
    block_mode=LanguageTableBlockVariants("BLOCK_8"),
    reward_factory=BlockToBlockReward,
    seed=42,
)

policy = SmolVLAPolicy(
    # Pulled from HF on first launch (cached under HF_HOME). To use a
    # different checkpoint, swap in another repo_id or a local path.
    checkpoint_path="mateoguaman/smolvla_lt_combined_sim_93185",
)
print("Env and policy ready.")

## `run_command` — execute one natural-language command

Identical contract to the LAVA notebook: takes a string, runs `max_steps`
steps with that goal held constant, returns `(frames, total_reward, info)`.
Ignores the env's `done` flag because the env's preset reward is unrelated
to the goal we manually feed in.

In [ ]:
def run_command(command: str, max_steps: int = 96):
    policy.reset(num_envs=1)

    obs = env._compute_observation()
    frames = [env.render(mode="rgb_array")]
    total_reward = 0.0
    info = {}

    state_text = state_to_text(env._last_state)
    print(f"Current state:\n{state_text}\n")
    print(f"Command: {command}\n")

    for i in range(max_steps):
        actions = policy.predict(
            goals=[command],
            obs_list=[obs],
            active_mask=np.array([True]),
        )
        action = actions[0]

        # Ignore env `done` -- the env's preset task is unrelated to the
        # high-level command we are chaining here. We always run max_steps
        # and label success manually via the Recorder.
        obs, reward, _done, info = env.step(action)
        frames.append(env.render(mode="rgb_array"))
        total_reward += reward

    print(f"\nTotal reward: {total_reward:.3f}  |  Won: {info.get('won', False)}")
    print(f"Final state:\n{state_to_text(env._last_state)}")
    return frames, total_reward, info


def show_rollout(frames, fps=5):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.axis("off")
    img = ax.imshow(frames[0])
    ax.set_title(f"Rollout  ({len(frames)} frames)")

    def update(i):
        img.set_data(frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig, update, frames=len(frames),
        interval=1000 / fps, blit=True
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())

## Initial state inspection

In [ ]:
env.reset()
state = env._last_state

print(state_to_text(state))

plt.figure(figsize=(5, 5))
plt.imshow(env.render(mode="rgb_array"))
plt.axis("off")
plt.title("Initial state")
plt.tight_layout()
plt.show()

## Try a single command

Edit the string and re-run. Resets the env first so you start from the
seed-42 initial layout each time.

In [ ]:
env.reset()

# frames, reward, info = run_command("move to the red moon")
# frames, reward, info = run_command("move behind the green star, then push it to the blue cube")
frames, reward, info = run_command("push the red moon in the left corner")

show_rollout(frames)

## Recorder — chain sub-commands and log attempts

Same `Recorder` from the LAVA notebooks (it's policy-agnostic — it just
calls `run_fn`). Each `rec.run(cmd)` logs raw frames + meta to
`~/recordings/lamer/<task>/attempt_NNNN/`. The returned frames have banners
showing the high-level task and the current sub-command.

In [ ]:
from recorder import Recorder

env.reset()
rec = Recorder(task="draw a 1", env=env, run_fn=run_command)
rec.print_state()

plt.figure(figsize=(5, 5))
plt.imshow(env.render(mode="rgb_array"))
plt.axis("off")
plt.title(f"Initial state -- {rec.task}")
plt.tight_layout()
plt.show()

# Subsequent cells: rec.run(cmd) instead of run_command(cmd).
# When finished with this attempt:
#     rec.label(success=True, notes="optional note")
#     rec.reset()    # resets env + starts attempt_0002
#     rec.summary()  # aggregate stats for this task

In [ ]:
frames, reward, info = rec.run("move behind the red moon")
show_rollout(frames)

In [ ]:
frames, reward, info = rec.run("push the red moon to the top-left corner")
show_rollout(frames)

### Label the attempt and start a fresh one

In [ ]:
# rec.label(success=False, notes="moon overshot")
# rec.reset()
# rec.summary()

## Cleanup

Run this when you're done with the notebook to terminate the SmolVLA
server subprocess. (`atexit` handles it on kernel shutdown too, but
calling `close` explicitly frees the GPU sooner.)

In [ ]:
policy.close()